In [2]:
from google.colab import userdata
import os

# Colab 左侧 🔑 Secrets → 新建 DEEPSEEK_KEY
api_key = userdata.get('DEEPSEEK_API_KEY')

# 安装客户端
# !pip install openai -q

from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

In [4]:
# 模拟一段财报文字（真实使用时从 PDF/Wind 提取）
report_text = """
贵州茅台2024年实现营业收入1741亿元，同比增长15.7%；
归母净利润857亿元，同比增长15.4%；
毛利率92.1%，净利率49.2%。
直销渠道占比提升至45%，i茅台平台贡献营收约110亿元。
"""

# 构建 Prompt
system_prompt = """你是专业的金融分析师。
请从财报文字中提取关键信息，
输出格式：营收/净利润/毛利率/核心亮点/潜在风险，每项一行。"""

# 调用 API
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": report_text}
    ],
    temperature=0.3,
    max_tokens=500
)

# 取出结果
result = response.choices[0].message.content
print(result)


营收/1741亿元，同比增长15.7%  
净利润/857亿元，同比增长15.4%  
毛利率/92.1%  
核心亮点/直销渠道占比提升至45%，i茅台平台贡献营收约110亿元  
潜在风险/高端白酒市场增速放缓，渠道改革可能影响传统经销商关系


# 差的 system prompt（太模糊）
"你是金融助手，帮我分析报告。"

# 好的 system prompt（角色+格式+约束）
"""你是资深卖方研究员，专注A股消费行业。
分析财报时：
1. 只提取数字事实，不做主观判断
2. 指出数据异常（同比变化>30%需标注）
3. 输出 JSON 格式：{revenue, profit, margin, flags}
4. 无法确认的数据标注 [待核实]"""

In [11]:
report_text = """
1. 业绩纪要
1.1. 财务表现：
•全年/单季收入：
○前三季度营收231.9亿元，同比增14.9%
○Q3营收74.3亿元，同比增12.7%
•净利润：
○前三季度归母净利57.5亿元，同比增24.5%
○Q3归母净利13亿元，同比增9.5%
○Q3汇率损失1.5亿元
•自由现金流：
○前三季度经营现金流91.1亿元，同比增45.2亿元
○Q3经营现金流48.1亿元，含对外许可首付款
1.2. 宏观市场环境：
•国际化与研发成熟：
○海外启动**20+**项临床研究
○全球处于扩大研发布局阶段
1.3. 业务产品亮点：
•创新药与产品结构优化：
○创新药收入占比55%
○累计获批24款1类新药和5款2类新药
○Q3获8项上市申请受理，48个临床批件
○瑞维鲁胺等产品快速放量
•GLP-1及新药领域：
○GLP-1产品9531三期临床显示体重下降20%
○小分子GLP-1定位口服便捷治疗
○GLP-1产品安全性数据优异，覆盖**1500+**名患者
•研发投入与创新实力：
○研发团队5600人，**60%**硕士以上
○双抗组合23P19+36为全球1st in class
○PD-1+阿帕替尼组合显著延长生存期
•许可收入与市场扩展：
○对外许可收入确认为2.9亿美元首付款
○累计完成15笔对外许可交易，潜在交易额超270亿美元
1.4. 竞争格局:
•全球药物创新竞争:
○两款自免产品已上市，差异化显著
○双功能抗体扩展治疗领域空间
1.5. 重要战略举措：
•国际化战略推进：
○持续坚持创新和国际化双轮驱动战略
○借船出海与自主出海并行
1.6. 商业合作：
•合作拓展收入规模：
○与GSK、格兰玛达成合作
○Blackmark等合作首付款已到账
1.7. 股东主要事件：
•融资：
○Clara B轮融资或带来股权增值
2. 未来指引
2.1. 收入变动：
•未来创新药收入增长强劲
•产品上市热潮扩大商业版图
2.2. 利润预期：
•首付款及海外药物销售收入进一步增加
•汇率波动影响利润预期
2.3. 营业成本：
•CFO确认费用率管控目标
2.4. 未来产品和市场发展：
•GLP-1产品定位体重管理与糖尿病双适应症领域
•多款创新药首次参与医保谈判
2.5. 其他未来重点财务和战略指引：
•研发资源调配加强新药储备
•全球开展**20+**项临床研究拓展国际市场
3. Q&A总结
•双抗全球竞争力布局与更新
•GLP-1全球临床与融资推动
•新药医保谈判策略增加市场渗透
•仿制药出海收入占比探索
•汇率损失对净利影响披露

情绪
1. 正面
•中国首个自主研发EZH2抑制剂一类创新药“菲博斯他片”（艾瑞景）获批上市，同时二类新药“恒格列金、瑞格列汀二甲双胍缓释片”（瑞乐堂）亦于本季度获批，至此恒瑞医药在中国已获批24款1类创新药与5款2类创新药，创新药产品矩阵持续扩容。
•第三季度营业收入达74.3亿元，同比增长12.7%；前三季度归母净利润达57.5亿元，同比增长24.5%，利润增长主要得益于创新药销售收入与对外许可收入显著提升，彰显创新转型成效。
•前三季度经营活动现金流净额达91.1亿元，同比大幅增长98.5%，第三季度单季现金流净额达48.1亿元，同比增长47.5%，主因海外授权合作首付款增加，其中Q3收到JK与贝达合计超5亿元首付款，现金流质量显著优化。
•累计完成15笔海外授权交易，潜在总交易额超270亿美元，本季度与GSK、Braile及格兰玛达达成新合作，国际化成果持续兑现，彰显全球创新价值认可。
•GLP-1核心资产9531（注射用GLP-1/GCG双受体激动剂）全球III期临床顺利推进，6mg剂量下实现约20%体重下降，疗效优于竞品，为公司代谢领域重磅产品，优先级最高，美国IND已获受理。
•自免领域全球首创双抗“23P19+36”进入临床II期，靶向系统性红斑狼疮（SLE）与IgA肾病，技术平台成熟，具备国际竞争潜力，即将启动III期临床，有望成为全球首个同类药物。
•卡瑞利珠单抗联合阿帕替尼用于肝癌新辅助治疗成果登顶国际顶级期刊，并在ESMO年会发布46项研究成果，含4项口头报告及1项大会主席专题报告，标志着恒瑞创新成果获国际权威学术平台正式认可。
•2025年医保谈判参与产品数量创历史纪录，重磅新药包括**氟唑帕利、海曲泊帕乙醇胺、瑞凯西单抗（PCSK9抑制剂）**及卡瑞利珠单抗新增适应症，公司明确推动“上市即放量”策略，加速医保准入与双通道落地。
•创新药收入占比超55%，销售体系全面重构，设立“首席市场官”统筹全球上市策略，强化数字化营销、院外渠道（DTP药房、互联网医疗）与患者管理能力，推动“潜力驱动型”资源布局。
•组建专职海外临床团队并落地美国CMO能力，推进“造船出海”战略，未来目标包括海外自主申报上市、本地化商业团队建设乃至海外生产基地建设，实现从技术输出向全球化运营升级。
2. 负面
•网络收入环比下降：网络收入环比下降3%，尽管Spectrum X和NVLink交换机的收入增长，但整体网络业务未能持续扩大。英伟达预计该业务将在2025年第一季度恢复增长。
•游戏业务疲弱表现：游戏收入第四季度为25亿美元，环比下降22%，全年同比增长仅9%，显示出该领域扩张的限制以及第四季度供应限制对生产带来的阻力。
•地理市场受限：中国数据中心销售占数据中心总收入比例仍偏低，主要受到出口管制的影响。英伟达未来出货量在中国市场可能保持当前水平，面临较大政策风险。
•第四季度运营和非运营费用增加：2025财年第四季度运营费用增长9%，非运营费用增长11%，主要受到更高的工程开发成本和基础设施成本的影响。
•毛利率压力显现：第四季度毛利率为73%，非毛利率为73.5%，预计未来几季度毛利率将维持在70%左右，随着Blackwell架构的推广才有望改善至75%，但短期内毛利率压力依然存在。
•AI基础设施周期替换影响：公司提到许多客户仍在使用旧架构（如Volta、Pascal和Ampere），表明部分市场未能完全向新一代产品（如Blackwell）迁移，可能影响短期增长潜力。
•汽车市场增速有限：尽管增长幅度较大，汽车业务全年收入仅为17亿美元，预计2025年收入增长至50亿美元，但相对公司其他业务规模仍属偏小。
•税率预期升高：公司预期2025年第一季度税率和非税率为17%（正负1%），对盈利能力可能带来额外负面影响。



财务一致预测 (2024)	预测值	公布值	较预期
全年营业总收入（亿)
超出预期	269.8
18.23%	279.85
22.63%	10.1
4.40%
全年归母净利润（亿)
超出预期	59.8
28.09%	63.37
47.28%	3.6
19.19%
全年摊薄EPS
超出预期	0.9	1.0	0.1
财务数据
定量分析：
时间	指标	绝对值	同比(%)	环比(%)
2025Q3	累计营业总收入(亿)	231.9	14.9
2025Q3	单季营业总收入(亿)	74.3	12.7	-13.2
2025Q3	累计归母净利润(亿)	57.5	24.5
2025Q3	单季归母净利润(亿)	13.0	9.5	-49.5
2025Q3	每股收益-摊薄	0.9
定性分析：
时间	指标	分析
2025Q3	2025Q3净利率	略有下降
2025Q3	管理费用	增加
主营业务
定量分析：
时间	指标	绝对值	同比(%)	环比(%)
2025Q3	恒瑞医药获批上市的1类创新药数量	24款
2025Q3	恒瑞医药获批上市的2类创新药数量	5款
2025Q3	恒瑞医药上市申请受理数量	8项
2025Q3	恒瑞医药获得临床批件数量	48个
2025Q3	恒瑞医药被列入突破性治疗品种名单的药物数量	4款
2020至2025Q3	恒瑞医药对外许可交易数量	15笔
2020至2025Q3	恒瑞医药对外许可潜在总交易金额	超过270亿美元
2025Q3	恒瑞医药海外启动临床研究数量	超过20项
2025Q3	创新药销售收入占产品销售收入比重	55%
2025Q3	恒瑞医药自主创新产品数量	100多个
2025Q3	恒瑞医药临床开发项目数量	600多个
2025Q3	恒瑞医药全球开展临床研究项目数量	20多个
2025Q3	恒瑞医药近期签署的对外许可协议数量	15笔
2025Q3	恒瑞医药对外许可协议总金额	超过270亿美元
2025Q3	恒瑞医药欧洲肿瘤大会报告数量	46个，其中4个为大会MA报告
定性分析：
时间	指标	分析
2025Q3	9531产品全球三期临床试验体重下降效果	约20%的体重下降
2025Q3	恒瑞医药海外仿制药收入占总收入比例	尚未超过5%
2025Q3	恒瑞医药欧洲肿瘤大会展台设立	首次设立展台
经营数据
定量分析：
时间	指标	绝对值	同比(%)	环比(%)
2025Q3	恒瑞医药研发团队人数	5600人
2025Q3	恒瑞医药研发团队硕士及以上学历成员比例	60%
未来预期
定性分析：
时间	指标	分析
2026FY	恒瑞医药体重管理相关产品预计推出时间	2026年
2025FY	凯瑞利斯单抗账面股权投资外部评估	年报中公布评估结果

"""

system_prompt = """你是资深的买方研究员，专注于A股医药行业。
分析以上业绩交流会的时候：
1.只提取数字事实，不做主观判断
2.指出数据异常（同比变化>30%需标注）
3.输出JSON格式{revenue, profit, margin, flags, researchs, progress}
4.无法确认的数据标注[待核实]"""

response = client.chat.completions.create(
    model = "deepseek-chat",
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",  "content": report_text}
    ],
    temperature = 0.3,
    max_tokens = 1000
)
result = response.choices[0].message.content
print(result)

{
  "revenue": {
    "前三季度": 231.9,
    "前三季度同比": 14.9,
    "Q3": 74.3,
    "Q3同比": 12.7
  },
  "profit": {
    "前三季度归母净利": 57.5,
    "前三季度归母净利同比": 24.5,
    "Q3归母净利": 13.0,
    "Q3归母净利同比": 9.5,
    "Q3汇率损失": 1.5
  },
  "margin": {
    "前三季度经营现金流": 91.1,
    "前三季度经营现金流同比": 45.2,
    "Q3经营现金流": 48.1,
    "Q3经营现金流同比": 47.5,
    "创新药收入占比": 55
  },
  "flags": [
    "前三季度经营现金流同比变化98.5% (>30%)",
    "Q3经营现金流同比变化47.5% (>30%)",
    "前三季度归母净利同比变化24.5% (<30%)",
    "Q3归母净利同比变化9.5% (<30%)"
  ],
  "researchs": {
    "海外启动临床研究": "20+",
    "研发团队人数": 5600,
    "硕士以上占比": 60,
    "累计获批1类新药": 24,
    "累计获批2类新药": 5,
    "Q3上市申请受理": 8,
    "Q3临床批件": 48,
    "GLP-1产品9531体重下降": 20,
    "GLP-1产品覆盖患者": "1500+",
    "对外许可首付款": 2.9,
    "累计对外许可交易": 15,
    "潜在交易额": "270+",
    "双抗23P19+36": "全球1st in class",
    "PD-1+阿帕替尼": "显著延长生存期"
  },
  "progress": {
    "GLP-1产品9531": "全球III期临床推进，美国IND已受理",
    "双抗23P19+36": "临床II期，即将启动III期",
    "2025年医保谈判": "参与产品数量创历史纪录，包括氟唑帕利、海曲泊帕乙醇胺、瑞凯西单抗等",
    "海外战略": "借船出海与自主出海并行，

In [13]:
report_text = """
1. 业绩纪要
1.1. 财务表现：
•全年/单季收入：
○前三季度营收231.9亿元，同比增14.9%
○Q3营收74.3亿元，同比增12.7%
•净利润：
○前三季度归母净利57.5亿元，同比增24.5%
○Q3归母净利13亿元，同比增9.5%
○Q3汇率损失1.5亿元
•自由现金流：
○前三季度经营现金流91.1亿元，同比增45.2亿元
○Q3经营现金流48.1亿元，含对外许可首付款
1.2. 宏观市场环境：
•国际化与研发成熟：
○海外启动**20+**项临床研究
○全球处于扩大研发布局阶段
1.3. 业务产品亮点：
•创新药与产品结构优化：
○创新药收入占比55%
○累计获批24款1类新药和5款2类新药
○Q3获8项上市申请受理，48个临床批件
○瑞维鲁胺等产品快速放量
•GLP-1及新药领域：
○GLP-1产品9531三期临床显示体重下降20%
○小分子GLP-1定位口服便捷治疗
○GLP-1产品安全性数据优异，覆盖**1500+**名患者
•研发投入与创新实力：
○研发团队5600人，**60%**硕士以上
○双抗组合23P19+36为全球1st in class
○PD-1+阿帕替尼组合显著延长生存期
•许可收入与市场扩展：
○对外许可收入确认为2.9亿美元首付款
○累计完成15笔对外许可交易，潜在交易额超270亿美元
1.4. 竞争格局:
•全球药物创新竞争:
○两款自免产品已上市，差异化显著
○双功能抗体扩展治疗领域空间
1.5. 重要战略举措：
•国际化战略推进：
○持续坚持创新和国际化双轮驱动战略
○借船出海与自主出海并行
1.6. 商业合作：
•合作拓展收入规模：
○与GSK、格兰玛达成合作
○Blackmark等合作首付款已到账
1.7. 股东主要事件：
•融资：
○Clara B轮融资或带来股权增值
2. 未来指引
2.1. 收入变动：
•未来创新药收入增长强劲
•产品上市热潮扩大商业版图
2.2. 利润预期：
•首付款及海外药物销售收入进一步增加
•汇率波动影响利润预期
2.3. 营业成本：
•CFO确认费用率管控目标
2.4. 未来产品和市场发展：
•GLP-1产品定位体重管理与糖尿病双适应症领域
•多款创新药首次参与医保谈判
2.5. 其他未来重点财务和战略指引：
•研发资源调配加强新药储备
•全球开展**20+**项临床研究拓展国际市场
3. Q&A总结
•双抗全球竞争力布局与更新
•GLP-1全球临床与融资推动
•新药医保谈判策略增加市场渗透
•仿制药出海收入占比探索
•汇率损失对净利影响披露

情绪
1. 正面
•中国首个自主研发EZH2抑制剂一类创新药“菲博斯他片”（艾瑞景）获批上市，同时二类新药“恒格列金、瑞格列汀二甲双胍缓释片”（瑞乐堂）亦于本季度获批，至此恒瑞医药在中国已获批24款1类创新药与5款2类创新药，创新药产品矩阵持续扩容。
•第三季度营业收入达74.3亿元，同比增长12.7%；前三季度归母净利润达57.5亿元，同比增长24.5%，利润增长主要得益于创新药销售收入与对外许可收入显著提升，彰显创新转型成效。
•前三季度经营活动现金流净额达91.1亿元，同比大幅增长98.5%，第三季度单季现金流净额达48.1亿元，同比增长47.5%，主因海外授权合作首付款增加，其中Q3收到JK与贝达合计超5亿元首付款，现金流质量显著优化。
•累计完成15笔海外授权交易，潜在总交易额超270亿美元，本季度与GSK、Braile及格兰玛达达成新合作，国际化成果持续兑现，彰显全球创新价值认可。
•GLP-1核心资产9531（注射用GLP-1/GCG双受体激动剂）全球III期临床顺利推进，6mg剂量下实现约20%体重下降，疗效优于竞品，为公司代谢领域重磅产品，优先级最高，美国IND已获受理。
•自免领域全球首创双抗“23P19+36”进入临床II期，靶向系统性红斑狼疮（SLE）与IgA肾病，技术平台成熟，具备国际竞争潜力，即将启动III期临床，有望成为全球首个同类药物。
•卡瑞利珠单抗联合阿帕替尼用于肝癌新辅助治疗成果登顶国际顶级期刊，并在ESMO年会发布46项研究成果，含4项口头报告及1项大会主席专题报告，标志着恒瑞创新成果获国际权威学术平台正式认可。
•2025年医保谈判参与产品数量创历史纪录，重磅新药包括**氟唑帕利、海曲泊帕乙醇胺、瑞凯西单抗（PCSK9抑制剂）**及卡瑞利珠单抗新增适应症，公司明确推动“上市即放量”策略，加速医保准入与双通道落地。
•创新药收入占比超55%，销售体系全面重构，设立“首席市场官”统筹全球上市策略，强化数字化营销、院外渠道（DTP药房、互联网医疗）与患者管理能力，推动“潜力驱动型”资源布局。
•组建专职海外临床团队并落地美国CMO能力，推进“造船出海”战略，未来目标包括海外自主申报上市、本地化商业团队建设乃至海外生产基地建设，实现从技术输出向全球化运营升级。
2. 负面
•网络收入环比下降：网络收入环比下降3%，尽管Spectrum X和NVLink交换机的收入增长，但整体网络业务未能持续扩大。英伟达预计该业务将在2025年第一季度恢复增长。
•游戏业务疲弱表现：游戏收入第四季度为25亿美元，环比下降22%，全年同比增长仅9%，显示出该领域扩张的限制以及第四季度供应限制对生产带来的阻力。
•地理市场受限：中国数据中心销售占数据中心总收入比例仍偏低，主要受到出口管制的影响。英伟达未来出货量在中国市场可能保持当前水平，面临较大政策风险。
•第四季度运营和非运营费用增加：2025财年第四季度运营费用增长9%，非运营费用增长11%，主要受到更高的工程开发成本和基础设施成本的影响。
•毛利率压力显现：第四季度毛利率为73%，非毛利率为73.5%，预计未来几季度毛利率将维持在70%左右，随着Blackwell架构的推广才有望改善至75%，但短期内毛利率压力依然存在。
•AI基础设施周期替换影响：公司提到许多客户仍在使用旧架构（如Volta、Pascal和Ampere），表明部分市场未能完全向新一代产品（如Blackwell）迁移，可能影响短期增长潜力。
•汽车市场增速有限：尽管增长幅度较大，汽车业务全年收入仅为17亿美元，预计2025年收入增长至50亿美元，但相对公司其他业务规模仍属偏小。
•税率预期升高：公司预期2025年第一季度税率和非税率为17%（正负1%），对盈利能力可能带来额外负面影响。



财务一致预测 (2024)	预测值	公布值	较预期
全年营业总收入（亿)
超出预期	269.8
18.23%	279.85
22.63%	10.1
4.40%
全年归母净利润（亿)
超出预期	59.8
28.09%	63.37
47.28%	3.6
19.19%
全年摊薄EPS
超出预期	0.9	1.0	0.1
财务数据
定量分析：
时间	指标	绝对值	同比(%)	环比(%)
2025Q3	累计营业总收入(亿)	231.9	14.9
2025Q3	单季营业总收入(亿)	74.3	12.7	-13.2
2025Q3	累计归母净利润(亿)	57.5	24.5
2025Q3	单季归母净利润(亿)	13.0	9.5	-49.5
2025Q3	每股收益-摊薄	0.9
定性分析：
时间	指标	分析
2025Q3	2025Q3净利率	略有下降
2025Q3	管理费用	增加
主营业务
定量分析：
时间	指标	绝对值	同比(%)	环比(%)
2025Q3	恒瑞医药获批上市的1类创新药数量	24款
2025Q3	恒瑞医药获批上市的2类创新药数量	5款
2025Q3	恒瑞医药上市申请受理数量	8项
2025Q3	恒瑞医药获得临床批件数量	48个
2025Q3	恒瑞医药被列入突破性治疗品种名单的药物数量	4款
2020至2025Q3	恒瑞医药对外许可交易数量	15笔
2020至2025Q3	恒瑞医药对外许可潜在总交易金额	超过270亿美元
2025Q3	恒瑞医药海外启动临床研究数量	超过20项
2025Q3	创新药销售收入占产品销售收入比重	55%
2025Q3	恒瑞医药自主创新产品数量	100多个
2025Q3	恒瑞医药临床开发项目数量	600多个
2025Q3	恒瑞医药全球开展临床研究项目数量	20多个
2025Q3	恒瑞医药近期签署的对外许可协议数量	15笔
2025Q3	恒瑞医药对外许可协议总金额	超过270亿美元
2025Q3	恒瑞医药欧洲肿瘤大会报告数量	46个，其中4个为大会MA报告
定性分析：
时间	指标	分析
2025Q3	9531产品全球三期临床试验体重下降效果	约20%的体重下降
2025Q3	恒瑞医药海外仿制药收入占总收入比例	尚未超过5%
2025Q3	恒瑞医药欧洲肿瘤大会展台设立	首次设立展台
经营数据
定量分析：
时间	指标	绝对值	同比(%)	环比(%)
2025Q3	恒瑞医药研发团队人数	5600人
2025Q3	恒瑞医药研发团队硕士及以上学历成员比例	60%
未来预期
定性分析：
时间	指标	分析
2026FY	恒瑞医药体重管理相关产品预计推出时间	2026年
2025FY	凯瑞利斯单抗账面股权投资外部评估	年报中公布评估结果

"""

system_prompt = """你是风控合规专员。
从公告文本中识别：股权质押/减持/诉讼/监管函/业绩预警等风险事件，
按严重程度排序。"""

response = client.chat.completions.create(
    model = "deepseek-reasoner",
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",  "content": report_text}
    ],
    temperature = 0.3,
    max_tokens = 1000
)
result = response.choices[0].message.content
print(result)

根据公告文本分析，未识别出与恒瑞医药直接相关的股权质押、减持、诉讼、监管函或业绩预警等风险事件。公告中涉及的“负面”部分（如网络收入下降、游戏业务疲弱、中国数据中心受限等）与其他公司（英伟达）的财务表现混淆，与恒瑞医药无关。恒瑞医药自身财务表现稳健，创新药收入增长，现金流良好，无系统性风险信号。因此，风险事件列表为空。
